In [ ]:
import pandas as pd
import os
from tqdm import tqdm
import numpy as np
import re

def analizza_riempimento(df, label):
    print(f"\n--- {label} ---")
    conteggio = df.count()
    totale_righe = len(df)
    for colonna, valori in conteggio.items():
        percentuale = (valori / totale_righe) * 100 if totale_righe > 0 else 0
        print(f"Colonna: {colonna:<30} | Presenti: {valori:>6} / {totale_righe:>6} ({percentuale:.2f}%)")
    print("-" * 60)

path_1 = r"C:\Data\Master_Database.xlsx"
path_2 = r"C:\Data\Daily_Update.xlsx"

df_1 = pd.read_excel(path_1)
df_1.columns = [col.strip().upper() for col in df_1.columns]
df_1 = df_1.map(lambda x: x.upper() if isinstance(x, str) else x)

analizza_riempimento(df_1, "STATO RIEMPIMENTO DB MASTER")

df_2 = pd.read_excel(path_2)
df_2.columns = [col.strip().upper() for col in df_2.columns]
df_2 = df_2.map(lambda x: x.upper() if isinstance(x, str) else x)

mappatura_colonne = {
    'ORIGINAL_COL_A': 'CATEGORY_TYPE',
    'ORIGINAL_COL_B': 'UNIQUE_ID',
    'ORIGINAL_COL_C': 'CAPACITY_VALUE',
    'ORIGINAL_COL_D': 'ITEM_DESCRIPTION',
    'ORIGINAL_COL_E': 'POINT_ID',
    'ORIGINAL_COL_F': 'ADDRESS',
    'ORIGINAL_COL_G': 'CITY',
    'ORIGINAL_COL_H': 'LATITUDE',
    'ORIGINAL_COL_I': 'LONGITUDE',
    'ORIGINAL_COL_J': 'CONTRACT_ID',
    'ORIGINAL_COL_K': 'USER_NAME',
    'ORIGINAL_COL_L': 'USER_TYPE',
    'ORIGINAL_COL_M': 'EQUIPMENT_DESC',
    'ORIGINAL_COL_N': 'START_DATE',
    'ORIGINAL_COL_O': 'END_DATE',
    'ORIGINAL_COL_P': 'ASSIGNMENT_DATE',
    'ORIGINAL_COL_Q': 'RELEASE_DATE',
    'ORIGINAL_COL_R': 'NOTES',
    'ORIGINAL_COL_S': 'POINT_TYPE'
}

df_2 = df_2[list(mappatura_colonne.keys())]
df_2 = df_2.rename(columns=mappatura_colonne)

df_2_duplicati = df_2[df_2.duplicated(subset=['UNIQUE_ID'], keep=False)]
df_2 = df_2[~df_2['UNIQUE_ID'].isin(df_2_duplicati['UNIQUE_ID'])]

df_1_2_union = pd.concat([df_1, df_2], ignore_index=True)
df_final = df_1_2_union.drop_duplicates(subset=['UNIQUE_ID'], keep='last').reset_index(drop=True)

analizza_riempimento(df_final, "STATO RIEMPIMENTO DB AGGIORNATO")

print(f"Aggiornamento completato. Record univoci totali: {len(df_final)}.")
display(df_final.head())

duplicati_check = df_final[df_final.duplicated(subset=['UNIQUE_ID'], keep=False)]
duplicati_check = duplicati_check.sort_values(by='UNIQUE_ID')
count_duplicati_tot = len(duplicati_check)
count_duplicati_unici = duplicati_check['UNIQUE_ID'].nunique()

print(f"--- ANALISI INTEGRITÀ DATI (UNIQUE_ID) ---")
print(f"ID duplicati rilevati: {count_duplicati_unici}")
print(f"Righe totali coinvolte: {count_duplicati_tot}")
print("-" * 40)

if count_duplicati_tot > 0:
    print(duplicati_check['UNIQUE_ID'].value_counts())
    display(duplicati_check.head(10))
else:
    print("Nessun duplicato rilevato. Integrità garantita.")

df_subset_reference = df_1.loc[
    df_1['CITY'] == 'REGION_A', 
    ['UNIQUE_ID', 'AREA_ZONE', 'CONTRACT_ID', 'ADDRESS', 'CITY']
].copy()

df_merged_full = df_final.merge(
    df_subset_reference[['UNIQUE_ID', 'CONTRACT_ID', 'ADDRESS', 'CITY', 'AREA_ZONE']], 
    on=['UNIQUE_ID', 'CONTRACT_ID', 'ADDRESS', 'CITY'], 
    how='left'
)

df_merged_full['AREA_ZONE'] = df_merged_full['AREA_ZONE_y'].combine_first(df_merged_full['AREA_ZONE_x'])
df_merged_full = df_merged_full.drop(columns=['AREA_ZONE_x', 'AREA_ZONE_y'])

perc_area = (df_merged_full['AREA_ZONE'].count() / len(df_merged_full)) * 100
print(f"Percentuale di copertura AREA_ZONE: {perc_area:.2f}%")

df_merged_full['CONTRACT_ID'] = df_merged_full['CONTRACT_ID'].replace('UNKNOWN', np.nan)

date_fields = ['RELEASE_DATE', 'ASSIGNMENT_DATE', 'START_DATE', 'END_DATE']
for col in date_fields:
    df_merged_full[col] = pd.to_datetime(df_merged_full[col], dayfirst=True, errors='coerce').dt.strftime('%d-%m-%Y')

mask_missing_capacity = df_merged_full['CAPACITY_VALUE'].isna()
extracted_numbers = df_merged_full.loc[mask_missing_capacity, 'EQUIPMENT_DESC'].str.extract(r'(\d+)')[0]
applied_values = extracted_numbers.dropna().apply(lambda x: f"{x} UNITS")
df_merged_full.loc[applied_values.index, 'CAPACITY_VALUE'] = applied_values

output_excel = r"C:\Data\Final_Database_Export.xlsx"

with pd.ExcelWriter(output_excel, engine='xlsxwriter') as writer:
    df_merged_full.to_excel(writer, index=False, sheet_name='Master_Data')
    
    workbook  = writer.book
    worksheet = writer.sheets['Master_Data']
    
    (max_row, max_col) = df_merged_full.shape
    column_settings = [{'header': column} for column in df_merged_full.columns]
    
    worksheet.add_table(0, 0, max_row, max_col - 1, {
        'columns': column_settings,
        'style': 'Table Style Medium 9'
    })
    
    for i, col in enumerate(df_merged_full.columns):
        column_len = df_merged_full[col].astype(str).str.len().max()
        column_len = max(column_len, len(col)) + 2
        worksheet.set_column(i, i, min(column_len, 50))

df_final_parquet = df_1.copy()
df_final_parquet['CONTRACT_ID'] = df_final_parquet['CONTRACT_ID'].replace("0", np.nan)
df_final_parquet['CONTRACT_ID'] = df_final_parquet['CONTRACT_ID'].fillna('').astype(str)

for col in df_final_parquet.select_dtypes(include=['object']).columns:
     df_final_parquet[col] = df_final_parquet[col].astype(str)

df_final_parquet.to_parquet("database_archive.parquet")

print(f"--- PROCESSO COMPLETATO ---")

In [ ]:
Riassunto del Progetto: Data Engineering & Automation Pipeline
Questo script Python implementa una pipeline completa di Data Cleaning, Integration e Enrichment per la gestione di anagrafiche complesse. Le principali funzionalità includono:

Standardizzazione Dati: Caricamento di dataset multipli (Excel), normalizzazione automatica dei nomi delle colonne (stripping e case normalization) e dei contenuti testuali per garantire la coerenza tra diverse sorgenti.

Deduplicazione Intelligente: Gestione dei record duplicati basata su identificatori univoci (UNIQUE_ID), assicurando che il database finale mantenga solo le informazioni più recenti e accurate.

Data Imputation tramite Regex: Recupero automatico di informazioni mancanti (es. capacità/volumi) estraendo pattern numerici da stringhe di testo non strutturato (descrizioni attrezzature) tramite espressioni regolari.

Join e Data Enrichment: Integrazione di metadati geografici e zonali attraverso operazioni di merge e combine_first, ottimizzando la copertura dei dati territoriali.

Output Multi-formato & Reporting:

Generazione di un file Excel professionale formattato nativamente come tabella, con auto-fit delle colonne per una leggibilità immediata.

Esportazione in formato Parquet, ottimizzato per storage ad alte prestazioni e analisi Big Data.

Monitoraggio in tempo reale della qualità del dato tramite report sulla percentuale di riempimento delle colonne (Data Quality Check).